# Evaluating cpu-time for each version of gl(r), built over time.


## File: `gl_computing_speed.ipynb`  
**Project:** HHG-SaDAS  

## Code Description  
This notebook contains various implementations of partial wave calculations developed over time, with an emphasis on improving efficiency and reducing computational cost. Different formulations of the radial and angular parts of the wavefunction are tested and compared, along with their numerical performance.  

---

**Author:** Siddhartha Mithiya  
**Affiliation:** Indian Institute of Technology (IIT) Mandi  
**License:** MIT License  
**Repository:** [https://github.com/Siddhartha-Acad/HHG-SaDAS.git](https://github.com/Siddhartha-Acad/HHG-SaDAS.git)  

---

## Notes  
- Compares different function implementations for accuracy, efficiency, and runtime scaling.  
- Demonstrates importance of vectorization, memory preallocation, and avoiding redundant computations in scientific code.  
- This notebook is part of the HHG-SaDAS package, developed during my MS(R) thesis:  
  *"Higher-Order Harmonic Generation and Harmonic-Power Enhancement in Noble-Gas Atoms Confined Inside C60".*  


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import legendre

In [2]:
def f(x):
    alpha = 2 * L_map / r_max
    return L_map * (1 + x) / (1 - x + alpha)

def f_p(x):
    alpha = 2 * L_map / r_max
    return L_map * (alpha + 2) / (1 - x + alpha)**2

In [3]:
n, l = 1, 0
N, L_map, r_max = 200, 80, 200                         # Radial Mapping parameters
L = 50                                                 # Angular mapping parameters

In [4]:
colloc_file = 'Algo-3_N=200_Colloc_pt_and_P_200_values.txt'
colloc_file = rf'E:\Python_programs\HHG\Thesis_figures\General_data\AnaDeriv_Colloc_pt\{colloc_file}'
colloc_file_data = np.loadtxt(colloc_file, skiprows=1)
colloc_pt = colloc_file_data[:, 0]

r = f(colloc_pt)                                       # Gauss-Lobatto collocation points

In [5]:
roots, weights = np.polynomial.legendre.leggauss(L+1)  # Gauss-Legendre Quadrature.
theta_k = np.arccos(roots)                             # Gauss-Legendre angular collocation points
len_r = len(r); len_k = len(weights)

In [6]:
A_file = 'H_GPSM_States__l=0_nos=10_N=200_rmax=200_Lmap=80.xlsx'
A_file = rf'E:\Python_programs\HHG\GPSM\GPSM_Y_l0\free_H\File_Generator\{A_file}'
A_data = pd.read_excel(A_file, header=None, skiprows=1).to_numpy().T

A_r = A_data[1:][n-1]
A_m = np.empty((len(theta_k), len(r)))
norm_fact = np.sqrt(N * (N + 1) / 2)
r_m, theta_m = np.meshgrid(r, theta_k)

for k in range(len(theta_k)):
    A_m[k] = A_r * np.sqrt((2*l+1) / (4*np.pi)) * legendre(l)(np.cos(theta_k[k]))

In [7]:
def gl_v1(Psi_t):
    """
    Compute partial waves g_l(r) for a given wavefunction using Gauss-Legendre quadrature.

    This function calculates the partial wave components g_l(r) of a wavefunction Psi_t(r, θ)
    using a projection onto Legendre polynomials evaluated at Gauss-Legendre nodes.

    Parameters
    ----------
    Psi_t : ndarray, shape (len_k, len_r)
        Complex-valued wavefunction evaluated at each θ_k and r_i point.
        Assumes θ-dependence has been discretized using Gauss-Legendre quadrature points (θ_k),
        and that Psi_t[k][i] = Ψ(r_i, θ_k) for each r_i and θ_k.

    Returns
    -------
    gl_arr : list of ndarray
        List of partial wave components g_l(r), one array per angular momentum index l (from 0 to L-1).
        Each array has shape (len_r,) and is complex-valued.

    Notes
    -----
    - Uses precomputed arrays: `roots`, `weights`, and global `L`, `len_k`, and `len_r`.
    - Legendre polynomials P_l(cos θ) are evaluated at the Gauss-Legendre nodes.
    - The projection uses the orthogonality of Legendre polynomials weighted by the Gauss-Legendre scheme.
    """
    gl_arr = []
    legendre_vals = np.array([[legendre(l_index)(root) for root in roots] for l_index in range(L+1)])
    for l_ind in range(L+1):
        g_l = np.zeros(len_r, dtype=np.complex128)
        psi_vals = np.array([[Psi_t[k][i] for k in range(len_k)] for i in range(len_r)])
        for i in range(len_r):
            g_l[i] = np.sum(weights * legendre_vals[l_ind, :] * psi_vals[i, :])
        gl_arr.append(g_l * np.sqrt(np.pi * (2 * l_ind + 1)))
    return gl_arr


In [8]:
def gl_v2(Psi_t):
    """
    Compute the partial wave projections g_l(r) of a given wavefunction using Gauss-Legendre quadrature.

    Parameters
    ----------
    Psi_t : ndarray of shape (len(theta), len(r))
        The full angular-radial wavefunction ψ(θ, r) at a fixed time. It must include
        function values at the Gauss-Legendre quadrature points θ_k, which are identified
        by `theta_k_locs`. This function expects:
            Psi_t[theta_k_locs[k], i] = ψ(θ_k, r_i)

    Returns
    -------
    gl_arr : list of ndarray, each of shape (len_r,)
        A list of partial wave projections g_l(r) for l = 0 to L. Each element is a real or
        complex-valued array representing the radial component of the wavefunction for the
        corresponding angular momentum quantum number l.

    Notes
    -----
    - The projection is computed using:
        g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r)
      where the sum is evaluated using Gauss-Legendre quadrature weights `weights` and
      Legendre polynomial values `legendre_vals[l_ind, :]`.

    - This implementation constructs an intermediate (len_r × len_k) array and stores
      results in a Python list, making it less memory efficient for repeated calls.
    """

    gl_arr = []
    psi_vals = np.array([[Psi_t[k][i] for k in range(len_k)] for i in range(len_r)])
    for l_ind in range(L+1):
        g_l = (weights * legendre_vals[l_ind, :]) @ psi_vals.T
        gl_arr.append(g_l * np.sqrt(np.pi * (2*l_ind + 1)))
    return gl_arr

legendre_vals = np.array([[legendre(l_index)(root) for root in roots] for l_index in range(L+1)])     # used in gl(r)

In [9]:
def gl_v3(Psi_t):
    """
    Compute partial wave projections g_l(r) from a given wavefunction ψ(θ, r) using
    an efficient, memory-optimized approach with Gauss-Legendre quadrature.

    Parameters
    ----------
    Psi_t : ndarray of shape (len(theta), len(r))
        The angular-radial wavefunction ψ(θ, r) at a fixed time. It must include
        values at Gauss-Legendre quadrature points θ_k, identified by `theta_k_locs`.
        This function expects:
            Psi_t[theta_k_locs[k], i] = ψ(θ_k, r_i)

    Returns
    -------
    gl_arr : list of ndarray, each of shape (len_r,)
        A list of partial wave projections g_l(r) for l = 0 to L. Each element is
        a NumPy array representing the radial component of the wavefunction for the
        corresponding angular momentum quantum number l.

    Notes
    -----
    - This version avoids explicitly building the intermediate (len_r × len_k) matrix,
      leading to lower memory usage and faster execution than gl_v1.
    - The projection is computed as:
        g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r)
      where the sum is evaluated efficiently using `np.tensordot`.
    - The result is returned as a list, where each entry corresponds to a different l.
    """

    gl_arr = []
    for l_ind in range(L + 1):
        proj = np.tensordot(weights * legendre_vals[l_ind, :], Psi_t, axes=([0], [0]))
        gl_arr.append(proj * np.sqrt(np.pi * (2*l_ind + 1)))
    return gl_arr

legendre_vals = np.array([[legendre(l_index)(root) for root in roots] for l_index in range(L+1)])     # used in gl(r)

In [10]:
def gl_v4(Psi_t):
    """
    Efficient and memory-optimized computation of partial wave projections g_l(r)
    from a given angular-radial wavefunction using Gauss-Legendre quadrature.

    Parameters
    ----------
    Psi_t : ndarray of shape (len(theta), len(r))
        The angular-radial wavefunction ψ(θ, r) at a fixed time. It must include
        values at Gauss-Legendre quadrature points θ_k, located at indices `theta_k_locs`.
        That is:
            Psi_t[theta_k_locs[k], i] = ψ(θ_k, r_i)

    Returns
    -------
    gl_arr : ndarray of shape (L+1, len_r)
        A NumPy array where each row contains the radial projection g_l(r) for a
        specific angular momentum quantum number l, ranging from 0 to L.

    Notes
    -----
    - This version avoids Python lists and appending, using a preallocated array for
      improved speed and cache efficiency.
    - The partial wave projection is computed as:
        g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r)
      using Gauss-Legendre quadrature weights `weights` and precomputed Legendre
      values `legendre_vals[l_ind, :]`.
    - The output is suitable for high-performance applications, especially when
      called repeatedly (e.g., in time evolution loops).
    """

    gl_arr = np.empty((L + 1, len_r), dtype=np.complex128)
    for l_ind in range(L + 1):
        gl_arr[l_ind] = (np.sqrt(np.pi * (2 * l_ind + 1)) *
                         np.tensordot(weights * legendre_vals[l_ind, :], Psi_t, axes=([0], [0])))
    return gl_arr


In [11]:
def gl_v5(Psi_t, gl_arr=None):
    """
    Compute partial wave projections g_l(r) from a given angular-radial wavefunction
    using Gauss-Legendre quadrature, with support for memory reuse.

    Parameters
    ----------
    Psi_t : ndarray of shape (len(theta), len(r))
        The angular-radial wavefunction ψ(θ, r) at a fixed time. It must include values
        at Gauss-Legendre quadrature points θ_k, identified by `theta_k_locs`.
        That is:
            Psi_t[theta_k_locs[k], i] = ψ(θ_k, r_i)

    gl_arr : ndarray of shape (L+1, len_r), optional
        Optional output array to store the result. If provided, it will be filled in-place
        and returned, avoiding repeated memory allocations. If None, a new array will be allocated.

    Returns
    -------
    gl_arr : ndarray of shape (L+1, len_r)
        The radial partial wave projections g_l(r) for l = 0 to L. Each row corresponds to
        a different angular momentum component.

    Notes
    -----
    - This function is designed for use in performance-critical code, such as loops over
      time steps, where memory allocation cost can become significant.
    - The projection is computed via:
        g_l(r) = √[π (2l + 1)] × ∑_k w_k P_l(cosθ_k) ψ(θ_k, r)
      using Gauss-Legendre quadrature with weights `weights` and precomputed
      Legendre values `legendre_vals[l, :]`.

    Example
    -------
    >> gl_empty = np.empty((L + 1, len_r), dtype=np.complex128)
    >> for t in range(num_steps):
    >>     update_Psi_t(...)
    >>     gl_vals = gl_v4(Psi_t, gl_empty)    # memory is reused each time
    """

    if gl_arr is None:
        gl_arr = np.empty((L + 1, len_r), dtype=np.complex128)

    for l_ind in range(L + 1):
        gl_arr[l_ind] = (np.sqrt(np.pi * (2 * l_ind + 1)) *
                         np.tensordot(weights * legendre_vals[l_ind, :], Psi_t, axes=([0], [0])))
    return gl_arr

In [12]:
%timeit gl_v1(A_m)

812 ms ± 16.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%timeit gl_v2(A_m)

5.38 ms ± 43.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
%timeit gl_v3(A_m)

3.29 ms ± 92.2 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [15]:
%timeit gl_v4(A_m)

3.39 ms ± 134 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
gl_empty = np.empty((L + 1, len_r), dtype=np.complex128)

In [17]:
%timeit gl_v5(A_m, gl_empty)

3.3 ms ± 91.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
